# Примеры запросов к SQLite-базе `hr_sample.sqlite`

База собрана из `sample_data.parquet` скриптом `build_sqlite.py`.

- Одна таблица `employees` со всеми 54 полями из `db_info.py`.
- Массивные колонки (`unit_id_tree`, `all_skills`, `all_succesors`, `children_*`, `lang_with_level`, `all_goals_desc`, `achievement_desc`) хранятся в виде JSON-строк — доступны через `json_extract` / `json_each`.
- `report_date` хранится как `YYYY-MM-DD` строка.

В репозитории лежит сэмпл на 5 000 сотрудников (~30 тыс. строк, ~40 МБ). Полную БД на 20 000 сотрудников можно собрать командой:

```bash
python build_sqlite.py --out hr.sqlite
```

In [1]:
import sqlite3
import pandas as pd

con = sqlite3.connect('hr_sample.sqlite')
pd.read_sql('SELECT name FROM sqlite_master WHERE type="table"', con)

,name
0,employees


## 1. Численность по месяцам

In [2]:
pd.read_sql('''
    SELECT report_date, COUNT(*) AS headcount
    FROM employees
    GROUP BY report_date
    ORDER BY report_date
''', con)

,report_date,headcount
0,2025-10-31,4669
1,2025-11-30,4674
2,2025-12-31,4662
3,2026-01-31,4668
4,2026-02-28,4676
5,2026-03-31,4671


## 2. Доля IT vs банкинг по блоку (L1 подразделения)

Вытаскиваем L1-uid из JSON-массива `unit_id_tree`.

In [3]:
pd.read_sql('''
    SELECT
        json_extract(unit_id_tree, '$[0]') AS l1_uid,
        SUBSTR(full_oshs_name, 1, INSTR(full_oshs_name, ' /') - 1) AS l1_name,
        COUNT(*) AS headcount
    FROM employees
    WHERE report_date = (SELECT MAX(report_date) FROM employees)
    GROUP BY l1_uid, l1_name
    ORDER BY headcount DESC
''', con)

,l1_uid,l1_name,headcount
0,1063,Блок Розничный Бизнес,1492
1,1115,Блок Корпоративно-Инвестиционный,1045
2,1001,Блок Технологии и Данные,938
3,1235,Блок Операционный,629
4,1170,Блок Риски и Комплаенс,567


## 3. Топ-10 трайбов по численности

In [4]:
pd.read_sql('''
    SELECT tribe_name, tribe_code, COUNT(*) AS headcount
    FROM employees
    WHERE report_date = (SELECT MAX(report_date) FROM employees)
      AND tribe_name IS NOT NULL
    GROUP BY tribe_name, tribe_code
    ORDER BY headcount DESC
    LIMIT 10
''', con)

,tribe_name,tribe_code,headcount
0,Трайб Express Retail Lending,5014.0,462
1,Tribe Back-Office Operations 7,5007.0,389
2,Tribe Investment Solutions 4,5004.0,343
3,Tribe Fraud Detection 6,5006.0,269
4,Трайб Nova Compliance Ops,5010.0,268
5,Трайб Unified Claims,5005.0,259
6,Трайб «Core Banking»,5009.0,219
7,Трайб «Data Governance»,5015.0,209
8,Трайб «Claims»,5001.0,186
9,Tribe SME Banking 17,5017.0,181


## 4. Распределение по грейдам и средняя оценка FP

In [5]:
pd.read_sql('''
    SELECT
        grade_num,
        COUNT(*) AS n,
        ROUND(AVG(CASE fp_res WHEN 'A' THEN 5 WHEN 'B' THEN 4 WHEN 'C' THEN 3
                              WHEN 'D' THEN 2 WHEN 'E' THEN 1 END), 2) AS avg_fp_res,
        ROUND(AVG(sber_los_cnt_days) / 365.0, 1) AS avg_tenure_years
    FROM employees
    WHERE report_date = (SELECT MAX(report_date) FROM employees)
    GROUP BY grade_num
    ORDER BY grade_num
''', con)

,grade_num,n,avg_fp_res,avg_tenure_years
0,7,575,3.24,7.3
1,8,581,3.27,7.2
2,9,613,3.28,7.0
3,10,577,3.29,7.5
4,11,739,3.28,7.6
5,12,695,3.30,7.2
6,13,770,3.28,7.5
7,14,116,3.41,7.9
8,15,4,2.75,6.9
9,16,1,2.00,8.0


## 5. Поиск сотрудников по навыку (разворачиваем JSON-массив)

Пример: сколько сотрудников владеют Python и какова их средняя оценка за результативность?

In [6]:
pd.read_sql('''
    SELECT
        COUNT(DISTINCT e.employee_id) AS python_users,
        ROUND(AVG(CASE e.fp_res WHEN 'A' THEN 5 WHEN 'B' THEN 4 WHEN 'C' THEN 3
                                WHEN 'D' THEN 2 WHEN 'E' THEN 1 END), 2) AS avg_fp_res
    FROM employees e, json_each(e.all_skills) s
    WHERE s.value = 'Python'
      AND e.report_date = (SELECT MAX(report_date) FROM employees)
''', con)

,python_users,avg_fp_res
0,114,3.46


## 6. Топ-15 самых популярных навыков

In [7]:
pd.read_sql('''
    SELECT s.value AS skill, COUNT(*) AS n
    FROM employees e, json_each(e.all_skills) s
    WHERE e.report_date = (SELECT MAX(report_date) FROM employees)
    GROUP BY s.value
    ORDER BY n DESC
    LIMIT 15
''', con)

,skill,n
0,Due diligence,894
1,Финансовое моделирование,890
2,Бухгалтерский учёт,890
3,Excel (продвинутый),872
4,Комплаенс,863
5,Переговоры,860
6,Оценка залогов,860
7,МСФО,859
8,1С,855
9,Презентации,852


## 7. Текучка: кто ушёл между первым и последним срезом

In [8]:
pd.read_sql('''
    WITH first_snap AS (SELECT MIN(report_date) d FROM employees),
         last_snap  AS (SELECT MAX(report_date) d FROM employees)
    SELECT
        (SELECT COUNT(*) FROM employees WHERE report_date = (SELECT d FROM first_snap)) AS at_start,
        (SELECT COUNT(*) FROM employees WHERE report_date = (SELECT d FROM last_snap))  AS at_end,
        (SELECT COUNT(*) FROM employees
         WHERE report_date = (SELECT d FROM first_snap)
           AND employee_id NOT IN (SELECT employee_id FROM employees
                                   WHERE report_date = (SELECT d FROM last_snap))) AS left_company,
        (SELECT COUNT(*) FROM employees
         WHERE report_date = (SELECT d FROM last_snap)
           AND employee_id NOT IN (SELECT employee_id FROM employees
                                   WHERE report_date = (SELECT d FROM first_snap))) AS hired
''', con)

,at_start,at_end,left_company,hired
0,4669,4671,316,318


## 8. История одного сотрудника (проверка реляционной непрерывности)

In [9]:
sample_id = pd.read_sql('''
    SELECT employee_id FROM employees
    GROUP BY employee_id HAVING COUNT(*) = (SELECT COUNT(DISTINCT report_date) FROM employees)
    ORDER BY RANDOM() LIMIT 1
''', con).iloc[0, 0]
print('employee_id =', sample_id)
pd.read_sql(f'''
    SELECT report_date, position_name, grade_num, sber_los_cnt_days,
           tribe_name, team_name, fp_res, fp_comp
    FROM employees WHERE employee_id = '{sample_id}'
    ORDER BY report_date
''', con)

employee_id = 10017146


,report_date,position_name,grade_num,sber_los_cnt_days,tribe_name,team_name,fp_res,fp_comp
0,2025-10-31,Руководитель отдела Pineapple-15,13,2867,Трайб Nova Compliance Ops,Команда Kiwi East,A,C
1,2025-11-30,Руководитель отдела Pineapple-15,13,2897,Трайб Nova Compliance Ops,Команда Kiwi East,A,C
2,2025-12-31,Руководитель отдела Pineapple-15,13,2928,Трайб Nova Compliance Ops,Команда Kiwi East,D,C
3,2026-01-31,Руководитель отдела Нептун-10,14,2959,Tribe Back-Office Operations 7,Команда Mango East,D,C
4,2026-02-28,Руководитель отдела Нептун-10,14,2987,Tribe Back-Office Operations 7,Команда Mango East,D,C
5,2026-03-31,Руководитель отдела Нептун-10,14,3018,Tribe Back-Office Operations 7,Команда Mango East,C,C


In [10]:
con.close()